# Session D — anticipatory action figures

Run: `python -m archive.impact_eda.impact_session_d --through 30`

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from processing_data.paths import EXT_DATA
from archive.impact_eda.impact_panel import _admin2_gdf

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_session_d.py").exists():
    ROOT = ROOT.parents[2]
OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG = OUT / "figures" / "session_d"
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", p.relative_to(ROOT))

In [ ]:
# Fig 1 — impact target vs validation
panel = pd.read_csv(OUT / "stage27_ground_truth_target.csv")
snap = panel.dropna(subset=["impact_score"]).sort_values("year").groupby("adm2_pcode").tail(1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
a = snap.dropna(subset=["people_affected"])
if len(a):
    axes[0].scatter(a.impact_score, np.log1p(a.people_affected), alpha=0.6, c="#2980b9")
    axes[0].set_xlabel("Impact score (exposure composite)")
    axes[0].set_ylabel("log1p OCHA people affected")
b = snap.dropna(subset=["disaster_share"])
if len(b):
    axes[1].scatter(b.impact_score, b.disaster_share, alpha=0.6, c="#c0392b")
    axes[1].set_xlabel("Impact score")
    axes[1].set_ylabel("DTM disaster arrival share")
fig.suptitle("SD-H1: exposure-anchored bad season vs humanitarian overlays (latest year per county)")
save(fig, "fig01_impact_target_definition.png")

In [ ]:
# Fig 2 — trigger skill curves
perf = pd.read_csv(OUT / "stage28_trigger_performance.csv")
fig, ax = plt.subplots(figsize=(6, 4))
for src, color in zip(
    ["climatology", "persistence", "oracle_wet_unusual"],
    ["#7f8c8d", "#27ae60", "#8e44ad"],
):
    sub = perf[perf.forecast_source == src]
    if sub.empty:
        continue
    ax.plot(sub.false_alarm_rate, sub.hit_rate, "o-", label=src, color=color, markersize=3)
ax.set_xlabel("False alarm rate")
ax.set_ylabel("Hit rate (bad season tercile)")
ax.legend(fontsize=8)
ax.set_title("SD-H2: trigger skill by forecast source")
save(fig, "fig02_trigger_skill_curves.png")

In [ ]:
# Fig 3 — cost-loss net value
cl = pd.read_csv(OUT / "stage28_cost_loss_curves.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for src in cl.forecast_source.unique():
    sub = cl[cl.forecast_source == src]
    ax.plot(sub.cost_loss_ratio, sub.net_value_units, "o-", label=src, markersize=4)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("Cost / loss ratio (C/L)")
ax.set_ylabel("Net value (arbitrary units)")
ax.legend(fontsize=7)
ax.set_title("Anticipatory action value vs C/L")
save(fig, "fig03_value_cost_surface.png")

In [ ]:
# Fig 4 — conflict access overlay (2024)
p24 = panel[panel.year == panel.year.max()].copy()
ged = pd.read_csv(OUT / "stage14_ged_admin2_month.csv")
g24 = ged[ged.year.isin([2023, 2024])].groupby("adm2_pcode")["n_events_all"].sum().reset_index()
p24 = p24.merge(g24, on="adm2_pcode", how="left")
p24["n_events_all"] = p24["n_events_all"].fillna(0)
admin2 = _admin2_gdf()
m = admin2.merge(p24, on="adm2_pcode", how="left")
m["log_pop"] = np.log1p(m["flood_exposed_pop"].fillna(0))
fig, ax = plt.subplots(figsize=(7, 8))
m.plot(column="log_pop", ax=ax, legend=True, cmap="Blues", edgecolor="gray", linewidth=0.3)
hi = m[m.n_events_all > m.n_events_all.median()]
hi.boundary.plot(ax=ax, edgecolor="#e74c3c", linewidth=1.2)
ax.set_title("Flood exposed pop (fill) + high-conflict counties outlined (2023–24 GED)")
ax.axis("off")
save(fig, "fig04_conflict_access_overlay.png")

In [ ]:
# Fig 5 — seasonal decision timeline
ged_m = ged.groupby("month")["n_events_all"].sum().reindex(range(1, 13), fill_value=0)
flood_m = pd.read_csv(OUT / "flood_admin2_month.csv")
u = flood_m[flood_m.mask_type == "unusual"].groupby("month")["pixel_days"].sum().reindex(range(1, 13), fill_value=0)
months = np.arange(1, 13)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(months - 0.2, u / u.max(), width=0.35, label="Unusual flood (norm)", color="#3498db")
ax1.bar(months + 0.2, ged_m / ged_m.max(), width=0.35, label="GED events (norm)", color="#e67e22")
ax1.axvspan(5, 6, color="#2ecc71", alpha=0.15, label="AA decision window (May–Jun)")
ax1.axvspan(6, 10.5, color="#9b59b6", alpha=0.1, label="Wet / flood peak")
ax1.set_xticks(months)
ax1.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax1.set_ylabel("Normalized intensity")
ax1.legend(loc="upper right", fontsize=8)
ax1.set_title("Seasonal timeline: decide early, act before peak; conflict lower in wet months")
save(fig, "fig05_seasonal_timeline_decision_window.png")